# Titanic Dataset Analysis Using Pandas

## Description

In this task, I analyze the Titanic dataset (`train.csv`) using pandas.

I will:
- Explore the dataset
- Clean missing values
- Perform analysis with groupby
- Filter meaningful subsets
- Draw insights about survival patterns

## Objectives
- Load and explore the Titanic dataset
- Handle missing values properly
- Analyze survival rates by gender, class, and age
- Filter specific passenger groups
- Provide insights for key questions

In [23]:
from google.colab import files
import pandas as pd

# Upload the file
uploaded = files.upload()

Saving Titanic-Dataset[1].csv to Titanic-Dataset[1] (1).csv


### Step 1: Dataset Exploration

I use:
- `.head()` to see first few rows
- `.info()` to understand columns and data types
- `.describe()` for statistical summary of numeric columns

In [24]:
import pandas as pd

# Load Titanic dataset
df = pd.read_csv("Titanic-Dataset[1].csv")

# Exploration
print("First 5 rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nStatistical Summary:")
print(df.describe())

First 5 rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN

### Step 2: Data Cleaning

- Age: replaced missing values with **median** (more realistic than 0)  
- Embarked: replaced missing values with **mode** (most common value)  
- Cabin: dropped due to too many missing values  
- Duplicates removed to ensure data accuracy

In [25]:
# Missing values before cleaning
print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill Age with median (assign back to avoid inplace warning)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fill Embarked with mode (assign back)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop Cabin column
df = df.drop(columns=['Cabin'])

# Remove duplicates
df = df.drop_duplicates()

# Verify missing values
print("\nMissing values after cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Missing values after cleaning:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


### Step 3: Data Analysis

- **groupby()** is used to analyze survival rates by gender, class, and age group  
- Age groups: Child (<12), Teen (12-18), Adult (19-59), Senior (60+)  
- This helps us understand which groups had higher survival probabilities

In [30]:
# Survival rate by gender
survival_by_gender = df.groupby('Sex')['Survived'].mean()
print("Survival rate by gender:\n", survival_by_gender)

# Survival rate by class
survival_by_class = df.groupby('Pclass')['Survived'].mean()
print("\nSurvival rate by class:\n", survival_by_class)

# Average age per class
average_age_per_class = df.groupby('Pclass')['Age'].mean()
print("\nAverage age per class:\n", average_age_per_class)

# Survival rate by age group
bins = [0, 12, 18, 59, 120]
labels = ['Child', 'Teen', 'Adult', 'Senior']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

survival_by_age_group = df.groupby('AgeGroup', observed=False)['Survived'].mean()
print("\nSurvival rate by age group:\n", survival_by_age_group)

Survival rate by gender:
 Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Survival rate by class:
 Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

Average age per class:
 Pclass
1    36.812130
2    29.765380
3    25.932627
Name: Age, dtype: float64

Survival rate by age group:
 AgeGroup
Child     0.579710
Teen      0.428571
Adult     0.365014
Senior    0.269231
Name: Survived, dtype: float64


### Step 4: Filtering

- Filtered **female survivors**  
- Filtered **children survivors** (Age < 12)  
- Filtered **1st class survivors** (high survival probability)  

In [27]:
# Female passengers who survived
female_survived = df[(df['Sex']=='female') & (df['Survived']==1)]

# Children who survived
children_survived = df[(df['Age']<12) & (df['Survived']==1)]

# 1st class survivors
first_class_survived = df[(df['Pclass']==1) & (df['Survived']==1)]

print("Number of female passengers who survived:", female_survived.shape[0])
print("Number of children who survived:", children_survived.shape[0])
print("Number of 1st class passengers who survived:", first_class_survived.shape[0])

Number of female passengers who survived: 233
Number of children who survived: 39
Number of 1st class passengers who survived: 136


### Step 5: Insights

- **Gender:** Females had higher survival rate than males  
- **Class:** Higher class (1st) increased chances of survival  
- **Children:** Had better survival chances than adults  
- **Highest survival combination:** Females in 1st class

In [28]:
print("--- Insights ---")

# Gender effect
if survival_by_gender['female'] > survival_by_gender['male']:
    print("Females were more likely to survive than males.")

# Class effect
print("Survival rate increases with class:\n", survival_by_class)

# Children
print("Children had higher survival rate than adults:\n", survival_by_age_group['Child'])

# Highest survival combination
highest_combo = df[(df['Sex']=='female') & (df['Pclass']==1) & (df['Survived']==1)]
print("Females in 1st class had the highest survival probability. Count:", highest_combo.shape[0])

--- Insights ---
Females were more likely to survive than males.
Survival rate increases with class:
 Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64
Children had higher survival rate than adults:
 0.5797101449275363
Females in 1st class had the highest survival probability. Count: 91


### Conclusion
Females and passengers in 1st class had the highest survival rates. Children were prioritized compared to adults. The combination with the highest survival probability was females in 1st class.